In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, WebSearchTool, ModelSettings
from IPython.display import Markdown, display
import asyncio
from pydantic import BaseModel


In [ ]:
load_dotenv(override=True)


In [ ]:
# Web search (grounds literature + references)
SEARCH_INSTRUCTIONS = (
    "You are a research assistant. Given a search term, search the web and "
    "produce a concise summary (2–3 paragraphs, under 300 words). "
    "Include specific page titles or URLs from results when the tool returns them. "
    "Output only the summary."
)

search_agent = Agent(
    name="Odinachi search",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)


# Validate topic
class ResponseFormat(BaseModel):
    is_valid: bool
    """Whether the topic is suitable for academic research"""
    reason: str
    """Reason for the validation"""
    topic: str
    """Working topic (refined from the user's input when needed)"""

validate_topic_agent = Agent(
    name="Validate Topic Agent",
    instructions="You are an academic researcher. Given a topic, validate if it is suitable for academic research; if not, suggest a closely related workable topic in `topic`.",
    output_type=ResponseFormat,
    model="gpt-4o-mini",
)


In [ ]:
# Abstract generation
abstract_agent = Agent(
    name="Abstract Generation Agent",
    instructions="You are an academic researcher. Given a topic, you generate a comprehensive abstract for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)


In [ ]:
# Introduction generation
introduction_agent = Agent(
    name="Introduction Generation Agent",
    instructions="You are an academic researcher. Given a topic, and a comprehensive abstract, you generate a comprehensive and convincing introduction for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)


In [ ]:
# Literature Review generation
literature_review_agent = Agent(
    name="Literature Review Generation Agent",
    instructions=(
        "You are an academic researcher. You receive abstract, introduction, and web research summaries. "
        "Synthesize a literature-style review grounded in those summaries; cite concrete sources when they appear there."
    ),
    output_type=str,
    model="gpt-4o-mini",
)

In [ ]:
# Methodology generation
methodology_agent = Agent(
    name="Methodology Generation Agent",
    instructions="You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, and a comprehensive literature review, you generate a comprehensive methodology for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)


In [ ]:
# Results / Findings generation
results_agent = Agent(
    name="Results Generation Agent",
    instructions=(
        "You are an academic researcher. Given prior sections, write a results section with detailed explanation. "
        "Use GitHub-flavored Markdown tables for structured numeric summaries. "
        "For diagrams, use ASCII sketches or fenced mermaid code blocks only (no binary images)."
    ),
    output_type=str,
    model="gpt-4o-mini",
)


In [ ]:
# Discussion and Conclusion generation
discussion_agent = Agent(
    name="Discussion Generation Agent",
    instructions="You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, a comprehensive literature review, a comprehensive methodology, and a comprehensive results section, you generate a comprehensive discussion section for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)


In [ ]:
# References / Bibliography generation
references_agent = Agent(
    name="References Generation Agent",
    instructions=(
        "You are an academic researcher. You are given the full draft sections plus web research summaries. "
        "Build a references list that prioritizes sources named in the web summaries; "
        "for purely synthetic framing, use clearly generic placeholders rather than fake DOIs."
    ),
    output_type=str,
    model="gpt-4o-mini",
)

In [ ]:
#validate the project generated


class ProjectEvaluation(BaseModel):
    well_defined_research_problem: int
    feasibility_and_planning: int
    methodological_rigor: int
    originality_and_contribution: int
    literature_review: int
    clear_structure_and_writing: int

evaluate_project_agent = Agent(
        name="Evaluate Project Agent",
        instructions="""
        You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, a comprehensive literature review, a comprehensive methodology, a comprehensive results section, and a comprehensive discussion section, you validate the project generated.
       
       Rate each criteria on a scale of 1 to 5, where 1 is the worst and 5 is the best.
       1. Well-Defined Research Problem: A good project starts with a clear, specific problem that warrants investigation, rather than just a broad topic.
       2. Feasibility and Planning: The project must be realistic, considering time constraints, availability of data, and resources.
       3. Methodological Rigor: Selecting an appropriate methodology—whether qualitative or quantitative—is crucial to produce valid, reliable, and unbiased results.
       4. Originality and Contribution: It should build upon existing scholarly literature to fill a research gap or offer a new interpretation of findings
       5. Literature Review: A comprehensive review demonstrates a thorough understanding of the current state of knowledge
       6. Clear Structure and Writing: The final project should be well-organized, typically including an introduction, literature review, methodology, results, discussion, and conclusion.

       The sum of all six scores will be used downstream (range 6–30); use the full 1–5 scale honestly.
""",
        output_type=ProjectEvaluation,
        model="gpt-4o-mini",
    )

In [ ]:
EVALUATION_THRESHOLD = 20  # sum of six 1–5 scores (range 6–30)
MAX_REGENERATION_ATTEMPTS = 2


def project_evaluation_sum(ev: ProjectEvaluation) -> int:
    return sum(ev.model_dump().values())


async def _web_summaries(topic: str) -> str:
    queries = [
        f"{topic} academic research overview",
        f"{topic} recent policy and society impact",
    ]
    tasks = [Runner.run(search_agent, f"Search term: {q}") for q in queries]
    results = await asyncio.gather(*tasks)
    parts = []
    for q, r in zip(queries, results):
        parts.append(f"### Query: {q}\n{r.final_output}")
    return "\n\n".join(parts)


async def _run_section(agent: Agent, prompt: str) -> str:
    result = await Runner.run(agent, prompt)
    return str(result.final_output)


async def generate_paper(
    initial_topic: str,
) -> tuple[str, ProjectEvaluation | None, ResponseFormat]:
    v = await Runner.run(validate_topic_agent, initial_topic)
    validation = v.final_output_as(ResponseFormat)
    topic = (validation.topic or initial_topic).strip()

    search_block = await _web_summaries(topic)

    paper = ""
    evaluation: ProjectEvaluation | None = None

    for attempt in range(MAX_REGENERATION_ATTEMPTS + 1):
        abstract = await _run_section(
            abstract_agent,
            f"Research topic:\n{topic}\n\nWrite the abstract only.",
        )
        introduction = await _run_section(
            introduction_agent,
            f"Topic:\n{topic}\n\nAbstract:\n{abstract}\n\nWrite the introduction only.",
        )
        literature = await _run_section(
            literature_review_agent,
            f"Topic:\n{topic}\n\nAbstract:\n{abstract}\n\nIntroduction:\n{introduction}\n\n"
            f"Web research summaries (ground claims in these; cite sources that appear here):\n{search_block}\n\n"
            f"Write the literature review only.",
        )
        methodology = await _run_section(
            methodology_agent,
            f"Topic:\n{topic}\n\nAbstract:\n{abstract}\n\nIntroduction:\n{introduction}\n\n"
            f"Literature review:\n{literature}\n\nWrite the methodology only.",
        )
        results = await _run_section(
            results_agent,
            f"Topic:\n{topic}\n\nAbstract:\n{abstract}\n\nIntroduction:\n{introduction}\n\n"
            f"Literature review:\n{literature}\n\nMethodology:\n{methodology}\n\n"
            f"Write the results section only.",
        )
        discussion = await _run_section(
            discussion_agent,
            f"Topic:\n{topic}\n\nAbstract:\n{abstract}\n\nIntroduction:\n{introduction}\n\n"
            f"Literature review:\n{literature}\n\nMethodology:\n{methodology}\n\nResults:\n{results}\n\n"
            f"Write the discussion and conclusion only.",
        )
        references = await _run_section(
            references_agent,
            f"Topic:\n{topic}\n\nAbstract:\n{abstract}\n\nIntroduction:\n{introduction}\n\n"
            f"Literature review:\n{literature}\n\nMethodology:\n{methodology}\n\nResults:\n{results}\n\n"
            f"Discussion:\n{discussion}\n\n"
            f"Web research summaries (prefer real titles/URLs from here for references):\n{search_block}\n\n"
            f"Write the references section only.",
        )

        paper = "\n\n".join(
            [
                f"# {topic}",
               "\n\n" + abstract,
                "\n\n" + introduction,
                "\n\n" + literature,
                "\n\n" + methodology,
                "\n\n" + results,
                "\n\n" + discussion,
                "\n\n" + references,
            ]
        )

        eval_prompt = (
            f"Topic:\n{topic}\n\nEvaluate the following draft paper.\n\n{paper}\n\n"
            "Return integers 1–5 only for each criterion."
        )
        ev_run = await Runner.run(evaluate_project_agent, eval_prompt)
        evaluation = ev_run.final_output_as(ProjectEvaluation)

        if project_evaluation_sum(evaluation) >= EVALUATION_THRESHOLD:
            break

    return paper, evaluation, validation

In [ ]:
topic = "What is the exact color of a person’s dreams?"

with trace("Generate Research Paper") as proj_trace:
    print(proj_trace.trace_id)
    paper_md, evaluation, validation = await generate_paper(topic)

if validation and not validation.is_valid:
    display(
        Markdown(
            f"**Topic refinement:** {validation.reason}\n\n"
            f"*Proceeding with the suggested topic from validation.*\n\n---\n"
        )
    )

display(Markdown(paper_md))
